In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Grundzustandsenergie-Schätzung vun der Heisenberg-Kette met VQE

*Schätzung för de Nutzung: Zwei Menute op enem Eagle r3 Prozessor (ANMERKUNG: Dat es nur e Schätzung. Ding Laufzick künnt anders sinn.)*

## Hintergrund

Dat Tutorial zeigt, wi mr e `Qiskit pattern` baut, deployt un laufe lööt för de Simulation vun ener Heisenberg-Kette un för de Schätzung vun de Grundzustandsenergie. Mieh Informatione övver `Qiskit patterns` un wi `Qiskit Serverless` benutzt weed för se en de Wolke ze deploye för verwaltete Usföhrung finget ehr op unser [Doku-Sigg övver IBM Quantum&reg; Platform](/guides/serverless).

## Vörussetzunge

Bevör mr met dämm Tutorial aanfange, stellt sescher, dat ihr dat Folgends installiert hätt:

* Qiskit SDK v1.2 oder neuer, met [Visualisierung](https://docs.quantum.ibm.com/api/qiskit/visualization) Ongerstötzung
* Qiskit Runtime v0.28 oder neuer (`pip install qiskit-ibm-runtime`)
* Qiskit Serverless (pip install qiskit_serverless)
* IBM Catalog (pip install qiskit-ibm-catalog)

## Setup

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2


def visualize_results(results):
    plt.plot(results["cost_history"], lw=2)
    plt.xlabel("Number of function evaluations")
    plt.ylabel("Energy")
    plt.show()

## Step 1: Map classical inputs to a quantum problem

*   Input: Number of spins
*   Output: Ansatz and Hamiltonian modeling the Heisenberg chain

Construct an ansatz and Hamiltonian that model a 10-spin Heisenberg chain. First, we import some generic packages and create some helper functions.

In [ ]:
num_spins = 10
ansatz = efficient_su2(num_qubits=num_spins, reps=2)

service = QiskitRuntimeService(
    channel="ibm_cloud",
    token="<YOUR_API_TOKEN>",  # Replace with your actual API token
    instance="<YOUR_INSTANCE_NAME>",  # Replace with your instance name if needed
)
backend = service.least_busy(
    operational=True, min_num_qubits=num_spins, simulator=False
)

coupling = backend.target.build_coupling_map()
reduced_coupling = coupling.reduce(list(range(num_spins)))

edge_list = reduced_coupling.graph.edge_list()
ham_list = []

for edge in edge_list:
    ham_list.append(("ZZ", edge, 0.5))
    ham_list.append(("YY", edge, 0.5))
    ham_list.append(("XX", edge, 0.5))

for qubit in reduced_coupling.physical_qubits:
    ham_list.append(("Z", [qubit], np.random.random() * 2 - 1))

hamiltonian = SparsePauliOp.from_sparse_list(ham_list, num_qubits=num_spins)

ansatz.draw("mpl", style="iqp")

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif" alt="Output of the previous code cell" />

## Schritt 1: Klassische Eingabe op e Quanteproblem mappe
*   Eingabe: Aanzahl vun Spins
*   Ausgabe: Ansatz un Hamiltonian för de Modellierung vun der Heisenberg-Kette

Baut ene Ansatz un Hamiltonian, de en 10-Spin Heisenberg-Kette modelliere. Zuerst importiere mer nen paar generische Pakete un maache e paar Hilfsfunktione.

In [5]:
target = backend.target
pm = generate_preset_pass_manager(optimization_level=3, target=target)
pm.scheduling = PassManager(
    [
        ALAPScheduleAnalysis(durations=target.durations()),
        PadDynamicalDecoupling(
            durations=target.durations(),
            dd_sequence=[XGate(), XGate()],
            pulse_alignment=target.pulse_alignment,
        ),
    ]
)
isa_ansatz = pm.run(ansatz)
isa_observable = hamiltonian.apply_layout(isa_ansatz.layout)
isa_ansatz.draw("mpl", scale=0.6, style="iqp", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif)

## Schritt 2: Problem för Quantehardware-Usföhrung optimiere

*   Eingabe: Abstrakte Schaltkreis, Observable
*   Ausgabe: Target-Schaltkreis un Observable, optimiert för de usgewählte QPU

Benutzt de `generate_preset_pass_manager` Funktioun us Qiskit för automatisch en Optimierungsroutine för unse Schaltkreis em Bezoch op de usgewählte QPU ze generiere. Mer wähle `optimization_level=3`, wat de höchste Level vun Optimierung vun de Preset-Pass-Manager es. Mer inklodiere och `ALAPScheduleAnalysis` un `PadDynamicalDecoupling` Scheduling-Passes för Dekohärenzfehler ze ongerdröcke.

In [ ]:
def spsa(
    fun, x0, args=(), A=30, alpha=0.9, a=0.3, c=0.1, gamma=0.4, maxiter=100
):
    nparams = len(x0)
    x = np.copy(x0)

    for i in range(maxiter):
        a_i = a / (A + i + 1) ** alpha
        c_i = c / (i + 1) ** gamma
        delta_i = np.random.choice([-1, 1], nparams)

        # two hardware calls
        eval_1 = fun(x + c_i * delta_i, *args)
        eval_2 = fun(x - c_i * delta_i, *args)

        # compute the gradient and update the parameters
        grad = (eval_1 - eval_2) / (2 * c_i) * np.reciprocal(delta_i)
        x = x - a_i * grad

    return x

In [ ]:
def cost_func(
    params: Sequence,
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
    cost_history_dict: dict,
) -> float:
    """Ground state energy evaluation."""
    energy = (
        estimator.run([(ansatz, hamiltonian, [params])]).result()[0].data.evs
    )

    cost_history_dict["iters"] += 1
    cost_history_dict["prev_vector"] = list(params)
    cost_history_dict["cost_history"].append(float(energy[0]))

    print(
        f"Fx Iters. done: {cost_history_dict['iters']} [Current cost: {round(energy[0], 5)}]",
        end="\r",
    )

    return energy


def solve(x0, isa_ansatz, isa_observable, maxiter=150):
    cost_history_dict = {
        "prev_vector": None,
        "iters": 0,
        "cost_history": [],
        "y_min": None,
    }

    # Evaluate the problem using a QPU via Qiskit IBM Runtime
    with Session(backend=backend) as session:
        estimator = EstimatorV2(mode=session)
        estimator.skip_transpilation = True
        x_opt = spsa(
            cost_func,
            x0=x0,
            args=(isa_ansatz, isa_observable, estimator, cost_history_dict),
            maxiter=maxiter,
        )

        y_min = cost_func(
            x_opt, isa_ansatz, isa_observable, estimator, cost_history_dict
        )

    return y_min, cost_history_dict

In [7]:
np.random.seed(42)
num_params = ansatz.num_parameters
params = 2 * np.pi * np.random.random(num_params)

## Et Qiskit-Muster en de Wolke deploye
För dat ze maache, verschüvt dä Source-Code ovve noh ener Datei, `./source/heisenberg.py`, packe der Code en e Skript, dat Eingabe entjejennemmp un de endgültige Lösung zeröckgitt, un laddet et schließlich op ene Remote-Cluster huh met der `QiskitFunction` Klass us `qiskit-ibm-catalog`. För Anleitunge övver et Spezifiziere vun externe Abhängigkeite, et Üvverjevve vun Eingabe-Argumente un mieh, luurt en de [Qiskit Serverless guides](/guides/serverless).

De Eingabe för et Pattern es de Aanzahl vun Spins en der Kette. De Ausgabe es en Schätzung vun der Grundzustandsenergie vum System.

In [8]:
maxiter = 50
spsa_min, spsa_history = solve(
    params, isa_ansatz, isa_observable, maxiter=maxiter
)

### Et Qiskit-Muster als verwaltete Service laufe lasse
Wenn mer et Pattern en de Wolke huhjelade han, künne mer et einfach met dem `QiskitServerless` Client laufe lasse.

In [10]:
print(f"Estimated ground state energy: {spsa_min}")

Estimated ground state energy: [-2.19621239]


In [11]:
results = {
    "spsa": spsa_history,
}

visualize_results(spsa_history)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/ecd7762a-0.avif" alt="Output of the previous code cell" />

## References

[1] Spall, J. C. (2002). Implementation of the simultaneous perturbation algorithm for stochastic optimization.
IEEE Transactions on Aerospace and Electronic Systems, 34(3), 817-823.

[2] Sahin, M. Emre, et al. (2025). Qiskit Machine Learning: an open-source library for quantum machine learning tasks at scale on quantum hardware and classical simulators. arXiv:2505.17756.

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:

- Find the ground-state energy of a sparse Hamiltonian by following the [SQD tutorial](/docs/tutorials/sample-based-quantum-diagonalization)
- Take the [Variational algorithm design](/learning/courses/variational-algorithm-design) course in IBM Learning
</Admonition>